# UIT-ViQuAD2.0 — Qwen2.5-1.5B DeLoRA

Notebook này chạy tuần tự một pipeline hoàn chỉnh:

1. Fine-tune **DeLoRA** → `qwen2.5-1.5b-instruct-delora-viquad2`
2. Đánh giá EM/F1/NoAns trên toàn bộ `validation`
3. Suy luận toàn bộ test và ghi `results.json`

DeLoRA ở đây là `peft.DeloraConfig` (`peft_type="DELORA"`), **không phải** DoRA (`use_dora=True`). DeLoRA phải nạp base model không lượng tử hóa vì bước khởi tạo tính norm trên trọng số gốc.

### Cấu hình RTX 3090 23 GB
- Micro-batch `4`, gradient accumulation `2` → effective batch `8`
- `max_seq_length=1024`, BF16 nếu GPU hỗ trợ, nếu không dùng FP16
- Tối đa 5 epochs; validation mỗi 200 optimizer steps
- Early stopping khi `eval_loss` không giảm ít nhất `0.001` trong 3 lần validation liên tiếp
- Tự khôi phục checkpoint có `eval_loss` thấp nhất

### Cách chạy
1. Chạy các pip cells, sau đó **Restart Kernel**.
2. Chạy tuần tự tất cả cells từ đầu đến cuối.
3. Sau training, notebook tự đánh giá DeLoRA và tạo:
   `qwen2.5-1.5b-instruct-delora-viquad2/results.json`
4. File submission có đúng dạng:
   `{"uit_id": "answer_of_model", ...}`; câu không có đáp án được ghi là chuỗi rỗng.

In [ ]:
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
# DeLoRA cần PEFT có DeloraConfig; pin đúng version đã tạo adapter ViCoQA thành công.
!pip install "peft==0.19.1" transformers trl accelerate bitsandbytes xformers datasets --no-cache-dir
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir
!pip install unsloth_zoo scikit-learn safetensors --no-cache-dir

In [ ]:
import importlib
import inspect

for pkg in ["torch", "transformers", "datasets", "unsloth", "peft"]:
    importlib.import_module(pkg)
    print(f"OK  {pkg}")

import peft
from peft import DeloraConfig

print(f"PEFT version: {peft.__version__}")
if peft.__version__ != "0.19.1":
    raise RuntimeError(f"Cần peft==0.19.1, hiện tại là {peft.__version__}. Restart Kernel sau pip cell.")

sig = inspect.signature(DeloraConfig.__init__).parameters
required = {"r", "delora_lambda", "target_modules", "module_dropout"}
missing = required.difference(sig)
if missing:
    raise RuntimeError(f"DeloraConfig thiếu tham số bắt buộc: {sorted(missing)}")
print("DeLoRA API OK:", ", ".join(k for k in sig if k != "self"))

In [ ]:
import warnings, logging, os, sys
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ACCELERATE_BYPASS_DEVICE_MAP"] = "true"
os.environ["ACCELERATE_NUM_PROCESSES"] = "1"
for _n in ("transformers", "datasets", "torch", "unsloth", "peft", "accelerate", "huggingface_hub"):
    logging.getLogger(_n).setLevel(logging.ERROR)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass
print("Warnings suppressed.")

In [ ]:
import json
import math
import gc
import re
import string
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ========== SHARED CONFIG (giữ cố định cho 3 method) ==========
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_NAME = "taidng/UIT-ViQuAD2.0"
NO_ANSWER_SENTINEL = "Không có đáp án trong đoạn văn"

SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện trong đoạn văn, không thêm từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm 'Đáp án:'.\n"
    f"3) Nếu không có đáp án trong đoạn văn, chỉ trả về: {NO_ANSWER_SENTINEL}"
)
USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn. Chỉ trả về cụm từ trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    f"Câu trả lời (span-only hoặc '{NO_ANSWER_SENTINEL}'):"
)

DATASET_ROOT = Path("../../")
DEV_JSON_PATH = DATASET_ROOT / "dev_viquad2.json"
TEST_JSON_PATH = DATASET_ROOT / "test_viquad2.json"
PROFILING_CONFIG_PATH = "profiling_config.json"
COMPARE_EVAL_PATH = "eval_compare_adapters_viquad2.json"

RUN_TRAINING = True          # train DeLoRA
RUN_METRIC_EVAL = True       # full validation sau khi train
RUN_SUBMISSION_EXPORT = True # full test 7301 → results.json
FORCE_HF_DATASET = True
FORCE_REEXPORT_JSON = True

# DeLoRA init tính norm(base.weight): bắt buộc base FP16/BF16, không dùng BnB 4/8-bit.
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
MAX_SEQ_CAP = 4096
MIN_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 32
USE_SPAN_POSTPROCESS = True

# Metric eval (EM/F1) — full validation có nhãn GT
EVAL_SPLIT = "validation"
EVAL_MAX_SAMPLES = None

# Submission results.json — BẮT BUỘC full test split
EXPECTED_TEST_SIZE = 7301
SUBMISSION_MAX_SAMPLES = None
SUBMISSION_LOG_EVERY = 50
EVAL_LOG_EVERY = 10
# Batch inference (chỉ ảnh hưởng tốc độ, greedy nên output KHÔNG đổi).
# RTX 3090/4090 24 GB + max_seq_length 1024: 16 an toàn; OOM thì hạ 8, dư VRAM thì thử 32.
INFER_BATCH_SIZE = 16
USE_UNSLOTH_FOR_INFERENCE = False  # True hay treo sau Adapter OK trên một số setup

# Eval/inference chạy TÁCH PROCESS (transformers thuần, KHÔNG unsloth) để tránh xung đột
# monkey-patch Qwen2Attention của Unsloth (lỗi 'apply_qkv') và loader offline thiếu weights.
# True  = MỌI method (lora/dora/tinylora) đều tách process — ổn định nhất (khuyến nghị).
# False = chỉ TinyLoRA tách process; LoRA/DoRA dùng Unsloth in-kernel (nhanh hơn nếu chạy được).
SUBPROCESS_EVAL_ALL = True

# RTX 3090 23 GB: batch=4/accum=2 (effective batch=8).
# Nếu OOM sau khi restart kernel: đổi lần lượt thành (2,4), rồi (1,8).
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Early stopping: patience đếm LẦN VALIDATION, không phải epoch.
USE_EARLY_STOPPING = True
MAX_TRAIN_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001
EVAL_STEPS = 200
SAVE_STEPS = 200
SAVE_TOTAL_LIMIT = 4

TRAIN_COMMON = dict(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_ratio=0.03,
    num_train_epochs=MAX_TRAIN_EPOCHS,
    learning_rate=2e-4,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
)

# Chỉ train/eval adapter DeLoRA; không chạy lại LoRA, TinyLoRA hoặc DoRA.
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# 4 variants — train tuần tự
ADAPTER_VARIANTS = [
    {
        "name": "lora",
        "save_path": "qwen2.5-1.5b-instruct-lora-viquad2",
        "output_dir": "outputs_viquad2_lora",
    },
    {
        "name": "tinylora",
        "save_path": "qwen2.5-1.5b-instruct-tinylora-viquad2",
        "output_dir": "outputs_viquad2_tinylora",
    },
    {
        "name": "dora",  # DoRA (use_dora=True qua Unsloth) — KHÁC với DeLoRA
        "save_path": "qwen2.5-1.5b-instruct-dora-viquad2",
        "output_dir": "outputs_viquad2_dora",
    },
    {
        "name": "delora",  # DeLoRA (peft.DeloraConfig, Frobenius-norm bounded) — cần peft>=0.19
        "save_path": "qwen2.5-1.5b-instruct-delora-viquad2",
        "output_dir": "outputs_viquad2_delora",
    },
]

# Bật/tắt từng method. Đang để CHỈ delora: train + eval variant MỚI, KHÔNG đụng 3 cái đã xong.
# Muốn chạy full lại: ["lora", "tinylora", "dora", "delora"]
TRAIN_METHODS = ["delora"]
EVAL_METHODS = ["delora"]

TQDM_BAR = "{desc}: {percentage:3.0f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"


def build_messages(context, question, answer=None, for_inference=False):
    user_content = USER_PROMPT_TEMPLATE.format(context=context, question=question)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    if answer is not None and not for_inference:
        messages.append({"role": "assistant", "content": answer})
    return messages


def sample_to_train_text(sample, tok):
    messages = build_messages(sample["context"], sample["question"], sample["answer"])
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def load_tokenizer(model_path=BASE_MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(model_path)
    if tok.chat_template is None:
        raise RuntimeError(f"Tokenizer {model_path} thiếu chat_template.")
    return tok


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

In [ ]:
def row_to_sample(row, *, allow_unlabeled=False):
    """Map HF row → sample; hidden test labels may be None OR empty answer lists."""
    context = str(row.get("context") or "").strip()
    question = str(row.get("question") or "").strip()
    if not context or not question:
        raise ValueError(f"Sample {row.get('uit_id') or row.get('id')} thiếu context/question")

    is_impossible = bool(row["is_impossible"])
    answers = row.get("answers")
    texts = (answers or {}).get("text") or []

    # HF test ẩn nhãn thường dùng answers={text: [], answer_start: []}, không chỉ None.
    if allow_unlabeled and not texts:
        return {
            "id": row.get("id", ""), "uit_id": row.get("uit_id", ""), "title": row.get("title", ""),
            "context": context, "question": question,
            "answer": None, "is_impossible": is_impossible, "has_label": False,
        }
    if answers is None:
        return {
            "id": row.get("id", ""), "uit_id": row.get("uit_id", ""), "title": row.get("title", ""),
            "context": context, "question": question,
            "answer": None, "is_impossible": is_impossible, "has_label": False,
        }
    if is_impossible or len(texts) == 0 or not str(texts[0]).strip():
        return {
            "id": row.get("id", ""), "uit_id": row.get("uit_id", ""), "title": row.get("title", ""),
            "context": row["context"], "question": row["question"],
            "answer": NO_ANSWER_SENTINEL, "is_impossible": True, "has_label": True,
        }
    return {
        "id": row.get("id", ""), "uit_id": row.get("uit_id", ""), "title": row.get("title", ""),
        "context": row["context"], "question": row["question"],
        "answer": texts[0], "is_impossible": False, "has_label": True,
    }


def hf_split_to_samples(split_dataset, *, allow_unlabeled=False):
    return [row_to_sample(row, allow_unlabeled=allow_unlabeled) for row in split_dataset]


def labeled_samples(samples):
    return [s for s in samples if s.get("has_label", True)]


def save_samples_json(samples, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(samples)} → {path.resolve()}")


def split_stats(name, samples):
    n = len(samples)
    n_no = sum(1 for s in samples if s["is_impossible"])
    print(f"{name}: {n} | NoAns: {n_no} ({100*n_no/max(n,1):.1f}%)")


try:
    raw_ds = load_dataset(DATASET_NAME)
    train_samples = hf_split_to_samples(raw_ds["train"])
    dev_samples = hf_split_to_samples(raw_ds["validation"])
    test_samples = hf_split_to_samples(raw_ds["test"], allow_unlabeled=True)
except Exception as e:
    if FORCE_HF_DATASET:
        raise RuntimeError(f"Không tải được HF: {e}")
    with open(TEST_JSON_PATH, encoding="utf-8") as f:
        test_samples = json.load(f)
    with open(DEV_JSON_PATH, encoding="utf-8") as f:
        dev_samples = json.load(f)
    train_samples = dev_samples

# Fail-fast để không train/eval trên split sai hoặc xuất submission thiếu/trùng key.
if len(test_samples) != EXPECTED_TEST_SIZE:
    raise RuntimeError(f"Test split phải có {EXPECTED_TEST_SIZE} câu, nhận {len(test_samples)}")
if any(s.get("has_label", True) for s in test_samples):
    raise RuntimeError("Test split ẩn nhãn nhưng có sample bị đánh dấu has_label=True")
test_keys = [s.get("uit_id") for s in test_samples]
if any(not isinstance(k, str) or not k for k in test_keys) or len(set(test_keys)) != len(test_keys):
    raise RuntimeError("Test split có uit_id rỗng, sai kiểu hoặc trùng")

if EVAL_SPLIT == "validation":
    eval_samples = labeled_samples(dev_samples)
elif EVAL_SPLIT == "test":
    eval_samples = labeled_samples(test_samples)
    if not eval_samples:
        raise RuntimeError(
            "HF test split không có nhãn answers. Đặt EVAL_SPLIT='validation' để eval có GT."
        )
else:
    raise ValueError(f"EVAL_SPLIT không hợp lệ: {EVAL_SPLIT}")

split_stats("Train", train_samples)
split_stats("Val", dev_samples)
split_stats("Test", test_samples)
split_stats(f"Eval ({EVAL_SPLIT})", eval_samples)
if FORCE_REEXPORT_JSON:
    save_samples_json(dev_samples, DEV_JSON_PATH)
    save_samples_json(test_samples, TEST_JSON_PATH)

In [ ]:
if "train_samples" not in globals():
    raise NameError("Chạy cell tải dataset trước.")


def compute_max_seq_length(samples, tok, cap=MAX_SEQ_CAP, min_len=MIN_SEQ_LENGTH):
    lengths = []
    total = len(samples)
    pbar = tqdm(samples, total=total, desc="Token profiling", unit="sample", bar_format=TQDM_BAR)
    for i, s in enumerate(pbar, 1):
        lengths.append(len(tok.encode(sample_to_train_text(s, tok))))
        pbar.set_postfix(done=f"{i}/{total}")
    lengths.sort()
    n = len(lengths)
    stats = {
        "min": lengths[0], "p50": lengths[n // 2],
        "p95": lengths[int(n * 0.95)], "p99": lengths[int(n * 0.99)], "max": lengths[-1],
    }
    chosen = max(((min(math.ceil(stats["p99"] * 1.05), cap) + 255) // 256) * 256, min_len)
    truncated = sum(1 for L in lengths if L > chosen)
    stats.update({"chosen_max_seq_length": chosen, "truncated_samples": truncated,
                  "truncated_pct": round(100 * truncated / n, 3)})
    return chosen, stats


tokenizer_prof = load_tokenizer()
if Path(PROFILING_CONFIG_PATH).exists() and not RUN_TRAINING:
    prof_cfg = json.load(open(PROFILING_CONFIG_PATH, encoding="utf-8"))
    max_seq_length = prof_cfg["max_seq_length"]
    length_stats = prof_cfg["token_length_stats"]
else:
    max_seq_length, length_stats = compute_max_seq_length(train_samples, tokenizer_prof)
    json.dump({"max_seq_length": max_seq_length, "token_length_stats": length_stats},
              open(PROFILING_CONFIG_PATH, "w", encoding="utf-8"), indent=2)
print(f"max_seq_length = {max_seq_length}")
for k, v in length_stats.items():
    print(f"  {k}: {v}")

In [ ]:
# Format dataset 1 lần — dùng chung cho cả 3 method
tokenizer_fmt = load_tokenizer()

def formatting_prompts_func(examples):
    texts = []
    for ctx, q, ans in zip(examples["context"], examples["question"], examples["answer"]):
        msgs = build_messages(ctx, q, ans)
        texts.append(tokenizer_fmt.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

train_hf = Dataset.from_list(train_samples)
dataset = train_hf.map(formatting_prompts_func, batched=True, remove_columns=train_hf.column_names)
print(f"Shared train dataset: {len(dataset)} samples")

eval_dataset = None
if USE_EARLY_STOPPING:
    dev_hf = Dataset.from_list(dev_samples)
    eval_dataset = dev_hf.map(formatting_prompts_func, batched=True, remove_columns=dev_hf.column_names)
    print(f"Eval dataset (early stopping): {len(eval_dataset)} samples")

print(dataset[0]["text"][:400])

## Train DeLoRA

Pipeline: load base FP16/BF16 → gắn `DeloraConfig` → train với early stopping → khôi phục best `eval_loss` checkpoint → lưu adapter và tokenizer.

`TRAIN_METHODS=["delora"]`, vì vậy các adapter LoRA/TinyLoRA/DoRA hiện có không bị train lại hoặc ghi đè.

In [ ]:
def apply_adapter(model, method_name):
    """Gắn đúng PEFT method lên base model đã load bằng Unsloth."""
    from unsloth import FastLanguageModel

    def _resolve_causallm(m):
        """TinyLoRA/PEFT cần ForCausalLM — KHÔNG unwrap .model (Qwen2Model)."""
        if hasattr(m, "prepare_inputs_for_generation"):
            return m
        if hasattr(m, "get_base_model"):
            base = m.get_base_model()
            if hasattr(base, "prepare_inputs_for_generation"):
                return base
        raise RuntimeError(
            "Không tìm thấy ForCausalLM. Đừng dùng model.model — đó là Qwen2Model "
            "(thiếu prepare_inputs_for_generation)."
        )

    if method_name == "lora":
        model = FastLanguageModel.get_peft_model(
            model,
            r=16, lora_alpha=32, target_modules=TARGET_MODULES,
            lora_dropout=0.05, bias="none",
            use_gradient_checkpointing="unsloth", random_state=3407,
            use_dora=False,
        )
        print("Applied: LoRA (r=16, alpha=32)")
        return model

    if method_name == "dora":
        model = FastLanguageModel.get_peft_model(
            model,
            r=16, lora_alpha=32, target_modules=TARGET_MODULES,
            lora_dropout=0.05, bias="none",
            use_gradient_checkpointing="unsloth", random_state=3407,
            use_dora=True,  # DoRA / Decomposed LoRA
        )
        print("Applied: DoRA (r=16, alpha=32, use_dora=True)")
        return model

    if method_name == "tinylora":
        import inspect
        import peft
        from peft import TinyLoraConfig, get_peft_model as peft_get_model

        # TinyLoRA (PEFT >= 0.19): r, u, tinylora_dropout — KHÔNG dùng lora_alpha/lora_dropout
        sig = inspect.signature(TinyLoraConfig.__init__).parameters
        desired = {
            "r": 2,                      # paper khuyến nghị r=2
            "u": 64,                     # modern API: trainable vector dim
            "num_projections": 64,       # legacy API alias của u
            "target_modules": TARGET_MODULES,
            "tinylora_dropout": 0.0,
            "lora_dropout": 0.0,         # chỉ dùng nếu PEFT cũ còn param này
            "bias": "none",
            "task_type": "CAUSAL_LM",
            "weight_tying": 0.0,
            "projection_seed": 3407,
            "init_weights": True,
        }
        tiny_kwargs = {k: v for k, v in desired.items() if k in sig}
        if "target_modules" not in tiny_kwargs:
            tiny_kwargs["target_modules"] = TARGET_MODULES

        try:
            tinylora_config = TinyLoraConfig(**tiny_kwargs)
        except Exception as e:
            raise RuntimeError(
                f"TinyLoRA không khả dụng với PEFT {peft.__version__}: {e}. "
                "Chạy lại pip cell (peft>=0.19) rồi Restart Kernel."
            ) from e

        causallm = _resolve_causallm(model)
        model = peft_get_model(causallm, tinylora_config)
        model.config.use_cache = False
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        model.print_trainable_parameters()
        u_val = tiny_kwargs.get("u", tiny_kwargs.get("num_projections", "?"))
        print(f"Applied: TinyLoRA (PEFT {peft.__version__}, r={tiny_kwargs.get('r')}, u={u_val})")
        return model

    if method_name == "delora":
        import peft
        from peft import DeloraConfig, get_peft_model as peft_get_model

        # DeLoRA (PEFT>=0.19): thay lora_alpha bằng delora_lambda (chặn trên Frobenius norm của
        # weight update), thay lora_dropout bằng module_dropout. Là PEFT method thuần — KHÔNG qua
        # Unsloth kernel (giống TinyLoRA), nên eval phải chạy tách process.
        delora_config = DeloraConfig(
            r=16,
            delora_lambda=15,
            target_modules=TARGET_MODULES,
            module_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            init_weights=True,
        )
        causallm = _resolve_causallm(model)
        model = peft_get_model(causallm, delora_config)
        model.config.use_cache = False
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        model.print_trainable_parameters()
        print(f"Applied: DeLoRA (PEFT {peft.__version__}, r=16, delora_lambda=15)")
        return model

    raise ValueError(f"Unknown method: {method_name}")


def train_one_variant(variant, max_seq_length, dataset, eval_dataset=None):
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import SFTTrainer
    from transformers import TrainingArguments, EarlyStoppingCallback
    import inspect
    import sys

    name = variant["name"]
    print("\n" + "=" * 60)
    print(f" TRAIN VARIANT: {name.upper()}")
    print("=" * 60)

    eval_on = USE_EARLY_STOPPING and eval_dataset is not None
    if USE_EARLY_STOPPING and eval_dataset is None:
        raise RuntimeError("USE_EARLY_STOPPING=True nhưng thiếu eval_dataset")
    if not torch.cuda.is_available():
        raise RuntimeError("Training Qwen2.5-1.5B chỉ được chạy trên CUDA GPU")

    clear_gpu()
    free_gib, total_gib = (x / 1024**3 for x in torch.cuda.mem_get_info())
    print(f"GPU preflight: free={free_gib:.2f}/{total_gib:.2f} GiB")
    if free_gib < 18.0:
        raise RuntimeError("Cần ít nhất 18 GiB VRAM trống. Restart kernel và đóng process GPU khác.")

    load_kwargs = dict(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=False,
        load_in_8bit=False,
    )
    if "device_map" in inspect.signature(FastLanguageModel.from_pretrained).parameters:
        load_kwargs["device_map"] = None
    model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
    model = model.to("cuda:0")
    model = apply_adapter(model, name)

    train_args = dict(
        **TRAIN_COMMON,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        output_dir=variant["output_dir"],
        disable_tqdm=False,
        logging_strategy="steps",
        logging_steps=SAVE_STEPS,
        logging_first_step=False,
        log_level="error",
        log_level_replica="error",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
        dataloader_num_workers=0,
    )
    ta_params = inspect.signature(TrainingArguments.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in ta_params else "evaluation_strategy"
    callbacks = []
    if eval_on:
        train_args.update({
            eval_key: "steps",
            "eval_steps": EVAL_STEPS,
            "load_best_model_at_end": True,
            "metric_for_best_model": "eval_loss",
            "greater_is_better": False,
        })
        callbacks.append(EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        ))
        print(
            f"Early stopping ON: max_epochs={MAX_TRAIN_EPOCHS}, patience={EARLY_STOPPING_PATIENCE}, "
            f"threshold={EARLY_STOPPING_THRESHOLD}, eval every {EVAL_STEPS} steps "
            f"on {len(eval_dataset)} val samples"
        )
    else:
        train_args[eval_key] = "no"
    train_args = {k: v for k, v in train_args.items() if k in ta_params or k == "output_dir"}

    sft_kwargs = dict(
        model=model,
        train_dataset=dataset,
        eval_dataset=eval_dataset if eval_on else None,
        args=TrainingArguments(**train_args),
        callbacks=callbacks,
    )
    sft_params = inspect.signature(SFTTrainer.__init__).parameters
    if "processing_class" in sft_params:
        sft_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sft_params:
        sft_kwargs["tokenizer"] = tokenizer
    for key, value in (
        ("dataset_text_field", "text"),
        ("max_seq_length", max_seq_length),
        ("dataset_num_proc", 1),
        ("packing", False),
    ):
        if key in sft_params:
            sft_kwargs[key] = value

    trainer = SFTTrainer(**sft_kwargs)
    cfg_cls, tr_cls = type(trainer.args), type(trainer)
    sys.modules["trl.trainer.sft_config"] = sys.modules[cfg_cls.__module__]
    sys.modules["trl.trainer.sft_trainer"] = sys.modules[tr_cls.__module__]
    sys.modules[cfg_cls.__module__].SFTConfig = cfg_cls
    sys.modules[tr_cls.__module__].SFTTrainer = tr_cls

    effective_batch = trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps
    steps_per_epoch = math.ceil(len(dataset) / max(effective_batch, 1))
    total_steps = int(steps_per_epoch * trainer.args.num_train_epochs)
    print(
        f"Planned: max_epochs={trainer.args.num_train_epochs}, steps/epoch≈{steps_per_epoch}, total_steps≈{total_steps}"
        + (f" | early_stop patience={EARLY_STOPPING_PATIENCE}" if eval_on else "")
    )

    train_result = trainer.train()
    print("Training metrics:", train_result.metrics)
    print("Best checkpoint:", trainer.state.best_model_checkpoint)
    print("Best eval_loss:", trainer.state.best_metric)

    save_path = Path(variant["save_path"])
    save_path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    saved_cfg = json.load(open(save_path / "adapter_config.json", encoding="utf-8"))
    if (saved_cfg.get("peft_type") or "").upper() != "DELORA":
        raise RuntimeError(f"Adapter lưu sai peft_type: {saved_cfg.get('peft_type')}")
    if saved_cfg.get("r") != 16 or saved_cfg.get("delora_lambda") != 15:
        raise RuntimeError("Adapter DeLoRA lưu sai r hoặc delora_lambda")
    print(f"Saved verified DeLoRA adapter → {save_path}")

    del trainer, model, tokenizer
    clear_gpu()
    return str(save_path)

In [ ]:
if RUN_TRAINING:
    variant_map = {v["name"]: v for v in ADAPTER_VARIANTS}
    trained_paths = {}
    for method in TRAIN_METHODS:
        if method not in variant_map:
            raise ValueError(f"Unknown TRAIN_METHODS item: {method}")
        path = train_one_variant(
            variant_map[method], max_seq_length, dataset,
            eval_dataset=eval_dataset if USE_EARLY_STOPPING else None,
        )
        trained_paths[method] = path
    print("\n✅ Train xong tất cả methods:")
    for k, v in trained_paths.items():
        print(f"  {k}: {v}")
else:
    print("RUN_TRAINING=False — bỏ qua train.")

## DeLoRA evaluation and submission

Đánh giá adapter DeLoRA trên validation (EM/F1/NoAns), sau đó suy luận full test và xuất `results.json` theo schema `{uit_id: answer}`.

In [ ]:
PREFIX_RE = re.compile(
    r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn)\s*[:\-]?\s*",
    re.IGNORECASE,
)

def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())

def compute_em(pred, truth):
    return int(normalize_text(pred) == normalize_text(truth))

def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt: return 1.0
    if not pt or not tt: return 0.0
    common = Counter(pt) & Counter(tt)
    n = sum(common.values())
    if n == 0: return 0.0
    p, r = n / len(pt), n / len(tt)
    return 2 * p * r / (p + r)

def is_no_answer(pred):
    return normalize_text(pred) == normalize_text(NO_ANSWER_SENTINEL)

def clean_prediction(raw):
    pred = raw.strip().split("\n")[0].strip().strip('"\'')
    return PREFIX_RE.sub("", pred).strip()


def align_to_context(pred, context):
    if not pred or is_no_answer(pred):
        return pred
    idx = context.lower().find(pred.lower())
    if idx >= 0:
        return context[idx:idx + len(pred)]
    words, pred_words = context.split(), normalize_text(pred).split()
    if not pred_words:
        return pred
    n = len(pred_words)
    best_span, best_f1 = pred, 0.0
    for i in range(len(words)):
        for j in range(i + 1, min(i + n + 4, len(words)) + 1):
            span = " ".join(words[i:j])
            f1 = compute_f1(span, pred)
            if f1 > best_f1:
                best_f1, best_span = f1, span
    return best_span.strip() if best_f1 >= 0.5 else pred


def _read_adapter_base_model(adapter_path):
    cfg_path = Path(adapter_path) / "adapter_config.json"
    if not cfg_path.exists():
        raise FileNotFoundError(f"Thiếu adapter_config.json trong {adapter_path}")
    cfg = json.load(open(cfg_path, encoding="utf-8"))
    return cfg.get("base_model_name_or_path", BASE_MODEL_NAME)


def _validate_adapter_files(adapter_path):
    adapter_path = Path(adapter_path)
    st_path = adapter_path / "adapter_model.safetensors"
    if not st_path.exists():
        raise FileNotFoundError(f"Thiếu {st_path}")

    head = open(st_path, "rb").read(200)
    if head.startswith(b"version https://git-lfs"):
        raise RuntimeError(
            f"{st_path.name} là Git LFS pointer — chưa pull weights thật. "
            "Chạy: git lfs pull"
        )

    size_mb = st_path.stat().st_size / (1024 * 1024)
    if size_mb < 0.1:
        raise RuntimeError(f"{st_path.name} quá nhỏ ({size_mb:.2f} MB) — file corrupt.")

    # Thử mở safetensors trước (bắt lỗi sớm)
    from safetensors import safe_open
    try:
        with safe_open(str(st_path), framework="pt", device="cpu") as f:
            _ = f.keys()
    except Exception as e:
        raise RuntimeError(f"{st_path.name} không đọc được: {e}") from e

    print(f"Adapter file OK: {st_path.name} ({size_mb:.1f} MB)")


def _adapter_peft_type(adapter_path):
    cfg = json.load(open(Path(adapter_path) / "adapter_config.json", encoding="utf-8"))
    return (cfg.get("peft_type") or "").upper()


def _preflight_peft_for_adapter(peft_type):
    """Fail-fast TRƯỚC khi tải base model: đảm bảo PEFT hỗ trợ peft_type của adapter.

    TinyLoRA/DeLoRA cần PEFT>=0.19 (`TinyLoraConfig` / `DeloraConfig`). Kiểm tra sớm để không
    tải base rồi mới crash lúc gắn adapter.
    """
    required = {"TINYLORA": "TinyLoraConfig", "DELORA": "DeloraConfig"}
    cls_name = required.get(peft_type)
    if not cls_name:
        return
    import peft
    try:
        getattr(__import__("peft", fromlist=[cls_name]), cls_name)
    except Exception as e:
        raise RuntimeError(
            f"[Preflight] Adapter peft_type={peft_type} nhưng PEFT {getattr(peft, '__version__', '?')} "
            f"KHÔNG có `{cls_name}` ({type(e).__name__}: {e}).\n"
            "→ Cài peft>=0.19 (có TinyLoRA/DeLoRA) rồi Restart Kernel."
        ) from e
    print(
        f"[Preflight] {peft_type} OK: `{cls_name}` khả dụng (PEFT {getattr(peft, '__version__', '?')}).",
        flush=True,
    )


def _load_eval_model_unsloth(adapter_path, base_model, max_seq_length):
    """LoRA / DoRA — Unsloth fast kernels (adapter chuẩn lora_A/lora_B)."""
    import torch
    from unsloth import FastLanguageModel
    from peft import PeftModel

    load_dtype = torch.float16 if not (LOAD_IN_4BIT or LOAD_IN_8BIT) else None
    print("[Eval] Loading base model on GPU (Unsloth)...", flush=True)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model,
        max_seq_length=max_seq_length,
        dtype=load_dtype,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_8bit=LOAD_IN_8BIT,
    )
    print("[Eval] Base loaded (Unsloth).", flush=True)

    # Chỉ load adapter weights trên CPU, giữ base trên GPU (không model.cpu())
    print("[Eval] Attaching adapter...", flush=True)
    model = PeftModel.from_pretrained(
        model, adapter_path, is_trainable=False, torch_device="cpu",
    )
    if USE_UNSLOTH_FOR_INFERENCE:
        print("[Eval] Unsloth for_inference...", flush=True)
        FastLanguageModel.for_inference(model)
    return model, tokenizer


def _load_eval_model_hf(adapter_path, base_model):
    """TinyLoRA — Unsloth kernel giả định lora_A/lora_B nên crash, và loader offline của
    Unsloth có thể không tìm thấy base weights (OSError: no model.safetensors).
    Dùng transformers thuần + PEFT (tải base online bình thường, đúng cho TinyLoRA)."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[Eval] Loading base model (HF plain, dtype={dtype})...", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=dtype).to(device)
    print("[Eval] Base loaded (HF plain).", flush=True)

    print("[Eval] Attaching adapter...", flush=True)
    model = PeftModel.from_pretrained(model, adapter_path, is_trainable=False).to(device)
    return model, tokenizer


def load_eval_model(adapter_path, max_seq_length):
    """LoRA/DoRA → Unsloth (fast). TinyLoRA → transformers thuần + PEFT (đúng & ổn định)."""
    _validate_adapter_files(adapter_path)
    base_model = _read_adapter_base_model(adapter_path)
    peft_type = _adapter_peft_type(adapter_path)
    print(f"Eval load: base={base_model} | adapter={adapter_path} | peft_type={peft_type}", flush=True)

    # Preflight: TinyLoRA cần PEFT có TinyLoraConfig — kiểm tra TRƯỚC khi tải base model
    _preflight_peft_for_adapter(peft_type)

    # LoRA/DoRA (peft_type=LORA) chạy Unsloth; TinyLoRA hoặc khác → HF thuần
    if peft_type == "LORA":
        model, tokenizer = _load_eval_model_unsloth(adapter_path, base_model, max_seq_length)
    else:
        model, tokenizer = _load_eval_model_hf(adapter_path, base_model)

    if not hasattr(model, "peft_config") or not model.peft_config:
        raise RuntimeError("Adapter KHÔNG được gắn.")
    print(f"Adapter OK: {list(model.peft_config.keys())}", flush=True)

    model.config.use_cache = True
    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("[Eval] Model ready — bắt đầu generate.", flush=True)
    return model, tokenizer

In [ ]:
ADAPTER_SUBMISSION_DIRS = {
    "lora": "qwen2.5-1.5b-instruct-lora-viquad2",
    "tinylora": "qwen2.5-1.5b-instruct-tinyLora-viquad2",
    "dora": "qwen2.5-1.5b-instruct-dora-viquad2",
    "delora": "qwen2.5-1.5b-instruct-delora-viquad2",
}


def predictions_to_submission_dict(predictions):
    """Strict submission schema: uit_id → model answer; no-answer sentinel → \"\"."""
    out = {}
    for row in predictions:
        key = row.get("uit_id")
        if not isinstance(key, str) or not key:
            raise RuntimeError(f"Prediction thiếu uit_id hợp lệ: {row.get('id')}")
        if key in out:
            raise RuntimeError(f"Prediction trùng uit_id: {key}")
        pred = (row.get("prediction") or "").strip()
        out[key] = "" if is_no_answer(pred) else pred
    return out


def _infer_predictions(method_name, adapter_path, samples, max_seq_length, *, log_every=10, desc=None):
    import time
    import sys
    from tqdm.auto import tqdm

    if not Path(adapter_path).exists():
        print(f"⚠ SKIP {method_name}: adapter path không tồn tại.", flush=True)
        return None

    # Mọi PEFT KHÔNG phải LORA (TinyLoRA, DeLoRA, ...) đều chạy TÁCH PROCESS transformers thuần,
    # KHÔNG unsloth. Unsloth patch toàn cục Qwen2Attention → HF thuần trong kernel sẽ lỗi 'apply_qkv'.
    # Đặt ở _infer_predictions nên áp dụng cho CẢ metric eval lẫn submission export.
    if SUBPROCESS_EVAL_ALL or _adapter_peft_type(adapter_path) != "LORA":
        return _infer_predictions_subprocess(
            method_name, adapter_path, samples, max_seq_length,
            log_every=log_every, desc=desc,
        )

    clear_gpu()
    model_eval, tokenizer_eval = load_eval_model(adapter_path, max_seq_length)
    model_eval.generation_config.max_length = None
    model_eval.generation_config.max_new_tokens = MAX_NEW_TOKENS

    total = len(samples)
    desc = desc or f"Infer-{method_name}"
    print(f"[Infer] {method_name}: {total} samples (max_seq={max_seq_length})", flush=True)

    preds, t0 = [], time.time()
    pbar = tqdm(samples, total=total, desc=desc, bar_format=TQDM_BAR, file=sys.stdout, mininterval=1.0)

    for i, s in enumerate(pbar, 1):
        msgs = build_messages(s["context"], s["question"], for_inference=True)
        prompt = tokenizer_eval.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer_eval(
            prompt, return_tensors="pt", truncation=True, max_length=max_seq_length,
        ).to(model_eval.device)

        if i == 1:
            print(f"[Infer] Warmup sample 1...", flush=True)

        with torch.no_grad():
            out = model_eval.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer_eval.pad_token_id,
                eos_token_id=tokenizer_eval.eos_token_id,
            )

        raw = tokenizer_eval.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred_raw = clean_prediction(raw)
        pred = align_to_context(pred_raw, s["context"]) if USE_SPAN_POSTPROCESS else pred_raw

        preds.append({
            "id": s["id"],
            "uit_id": s.get("uit_id", ""),
            "question": s.get("question", ""),
            "is_impossible": s.get("is_impossible"),
            "has_label": s.get("has_label", True),
            "ground_truth": s.get("answer"),
            "prediction_raw": pred_raw,
            "prediction": pred,
        })

        if i == 1 or i % log_every == 0 or i == total:
            elapsed = time.time() - t0
            rate = i / max(elapsed, 0.001)
            eta = (total - i) / max(rate, 0.001)
            print(f"[Infer] {i}/{total} | {elapsed/60:.1f}m | ETA {eta/60:.1f}m | {rate:.2f} sample/s", flush=True)

    pbar.close()
    print(f"[Infer] Done {total} samples in {(time.time()-t0)/60:.1f} min", flush=True)

    del model_eval, tokenizer_eval
    clear_gpu()
    return preds


def eval_one_adapter(method_name, adapter_path, eval_samples, max_seq_length):
    import random

    print(f"\n--- Metric eval: {method_name} | {adapter_path} ---", flush=True)

    if EVAL_MAX_SAMPLES:
        rng = random.Random(3407)
        has = [s for s in eval_samples if not s["is_impossible"]]
        no = [s for s in eval_samples if s["is_impossible"]]
        half = EVAL_MAX_SAMPLES // 2
        pick_has = rng.sample(has, min(half, len(has)))
        pick_no = rng.sample(no, min(EVAL_MAX_SAMPLES - len(pick_has), len(no)))
        samples = pick_has + pick_no
        rng.shuffle(samples)
        print(f"[Eval] Smoke test: {len(pick_has)} hasAns + {len(pick_no)} noAns", flush=True)
    else:
        samples = eval_samples

    preds = _infer_predictions(
        method_name, adapter_path, samples, max_seq_length,
        log_every=EVAL_LOG_EVERY, desc=f"Eval-{method_name}",
    )
    if preds is None:
        return None

    has_em, has_f1, no_ok = [], [], []
    for s, row in zip(samples, preds):
        pred = row["prediction"]
        if not s.get("has_label", True) or s.get("answer") is None:
            row["em"], row["f1"] = None, None
            continue
        if s["is_impossible"]:
            ok = int(is_no_answer(pred))
            no_ok.append(ok)
            row["em"], row["f1"] = ok, float(ok)
        else:
            em = compute_em(pred, s["answer"])
            f1 = compute_f1(pred, s["answer"])
            has_em.append(em)
            has_f1.append(f1)
            row["em"], row["f1"] = em, f1
            if len(has_em) <= 3:
                print(f"  [sample] GT='{s['answer']}' | pred='{pred[:60]}' | EM={em}", flush=True)

    n_has, n_no = len(has_em), len(no_ok)
    metrics = {
        "method": method_name,
        "adapter": adapter_path,
        "hasans_em": round(100 * sum(has_em) / max(n_has, 1), 4),
        "hasans_f1": round(100 * sum(has_f1) / max(n_has, 1), 4),
        "noans_accuracy": round(100 * sum(no_ok) / n_no, 4) if n_no else None,
        "n_hasans": n_has,
        "n_noans": n_no,
        "total": len(samples),
    }
    return {"metrics": metrics, "predictions": preds}


# ============ Eval inference TÁCH PROCESS (transformers thuần, KHÔNG unsloth) ============
# Unsloth monkey-patch toàn cục Qwen2Attention khi được import (cho LoRA/DoRA). Model nạp
# bằng transformers thuần trong cùng kernel sẽ crash 'apply_qkv'; loader offline của Unsloth
# còn có thể không thấy base weights. Chạy tách hẳn 1 process (interpreter mới, chưa import
# unsloth) là cách chắc chắn nhất — dùng chung cho cả metric eval và submission export.
_EVAL_INFER_SUBPROCESS_SRC = r'''"""Standalone eval inference — process riêng, KHÔNG import unsloth (trả về predictions)."""
from __future__ import annotations

import argparse
import json
import os
import re
import string
import sys
import time
import unicodedata
from collections import Counter
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONWARNINGS", "ignore")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

NO_ANSWER_SENTINEL = "Không có đáp án trong đoạn văn"

SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện trong đoạn văn, không thêm từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm 'Đáp án:'.\n"
    f"3) Nếu không có đáp án trong đoạn văn, chỉ trả về: {NO_ANSWER_SENTINEL}"
)
USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn. Chỉ trả về cụm từ trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    f"Câu trả lời (span-only hoặc '{NO_ANSWER_SENTINEL}'):"
)

PREFIX_RE = re.compile(
    r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn)\s*[:\-]?\s*",
    re.IGNORECASE,
)


def build_messages(context, question):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(context=context, question=question)},
    ]


def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())


def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt:
        return 1.0
    if not pt or not tt:
        return 0.0
    common = Counter(pt) & Counter(tt)
    n = sum(common.values())
    if n == 0:
        return 0.0
    p, r = n / len(pt), n / len(tt)
    return 2 * p * r / (p + r)


def is_no_answer(pred):
    return normalize_text(pred) == normalize_text(NO_ANSWER_SENTINEL)


def clean_prediction(raw):
    pred = raw.strip().split("\n")[0].strip().strip("\"'")
    return PREFIX_RE.sub("", pred).strip()


def align_to_context(pred, context):
    if not pred or is_no_answer(pred):
        return pred
    idx = context.lower().find(pred.lower())
    if idx >= 0:
        return context[idx:idx + len(pred)]
    words, pred_words = context.split(), normalize_text(pred).split()
    if not pred_words:
        return pred
    n = len(pred_words)
    best_span, best_f1 = pred, 0.0
    for i in range(len(words)):
        for j in range(i + 1, min(i + n + 4, len(words)) + 1):
            span = " ".join(words[i:j])
            f1 = compute_f1(span, pred)
            if f1 > best_f1:
                best_f1, best_span = f1, span
    return best_span.strip() if best_f1 >= 0.5 else pred


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--adapter-dir", required=True)
    p.add_argument("--samples-json", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--base-model", default="Qwen/Qwen2.5-1.5B-Instruct")
    p.add_argument("--max-seq-length", type=int, default=1024)
    p.add_argument("--max-new-tokens", type=int, default=32)
    p.add_argument("--max-samples", type=int, default=0)
    p.add_argument("--batch-size", type=int, default=16)
    p.add_argument("--log-every", type=int, default=50)
    p.add_argument("--no-span-postprocess", action="store_true")
    return p.parse_args()


def main():
    args = parse_args()
    if "unsloth" in sys.modules:
        raise RuntimeError("unsloth đã bị import — mất tác dụng cách ly process.")

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    try:
        from transformers.utils import logging as hf_logging
        hf_logging.set_verbosity_error()
    except Exception:
        pass

    adapter_dir = Path(args.adapter_dir)
    cfg = json.load(open(adapter_dir / "adapter_config.json", encoding="utf-8"))
    base_model = cfg.get("base_model_name_or_path") or args.base_model
    # Unsloth có thể ghi lại Hub id thành unsloth/... khi save; dùng Qwen id chuẩn để eval ổn định.
    if str(base_model).lower().startswith("unsloth/"):
        print(f"[Sub] remap saved base {base_model} → {args.base_model}", flush=True)
        base_model = args.base_model
    peft_type = (cfg.get("peft_type") or "").upper()
    if peft_type != "DELORA":
        raise RuntimeError(f"Expected peft_type=DELORA, got {peft_type!r}")
    _required_cfg = "DeloraConfig"
    if _required_cfg:
        try:
            getattr(__import__("peft", fromlist=[_required_cfg]), _required_cfg)
        except Exception as e:
            raise RuntimeError(f"PEFT không có {_required_cfg} ({type(e).__name__}: {e}).") from e

    samples = json.load(open(args.samples_json, encoding="utf-8"))
    if args.max_samples and args.max_samples > 0:
        samples = samples[: args.max_samples]
    total = len(samples)
    print(f"[Sub] base={base_model} | adapter={adapter_dir} | peft_type={peft_type} | samples={total}", flush=True)

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[Sub] Loading base (dtype={dtype}, device={device})...", flush=True)
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=dtype).to(device)
    model = PeftModel.from_pretrained(model, str(adapter_dir), is_trainable=False).to(device)
    if not getattr(model, "peft_config", None):
        raise RuntimeError("Adapter KHÔNG được gắn.")
    print(f"[Sub] Adapter OK: {list(model.peft_config.keys())}", flush=True)

    model.config.use_cache = True
    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Chỉ dùng max_new_tokens → tránh spam "Both max_new_tokens and max_length".
    model.generation_config.max_length = None
    model.generation_config.max_new_tokens = args.max_new_tokens
    for _k in ("temperature", "top_p", "top_k"):
        if hasattr(model.generation_config, _k):
            setattr(model.generation_config, _k, None)

    use_span = not args.no_span_postprocess
    # Batch inference: PHẢI left-padding để phần sinh mới nằm cùng vị trí cuối chuỗi.
    tokenizer.padding_side = "left"
    bs = max(1, args.batch_size)

    # Render prompt trước; sắp theo độ dài để giảm padding thừa trong mỗi batch.
    prompts = []
    for s in samples:
        msgs = build_messages(s["context"], s["question"])
        prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    order = sorted(range(len(samples)), key=lambda idx: len(prompts[idx]))

    preds = [None] * len(samples)
    t0 = time.time()
    done = 0
    for start in range(0, len(order), bs):
        batch_idx = order[start:start + bs]
        batch_prompts = [prompts[j] for j in batch_idx]
        enc = tokenizer(
            batch_prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=args.max_seq_length,
        ).to(device)
        with torch.no_grad():
            out = model.generate(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                max_new_tokens=args.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        gen = out[:, enc["input_ids"].shape[1]:]
        for k, j in enumerate(batch_idx):
            s = samples[j]
            raw = tokenizer.decode(gen[k], skip_special_tokens=True)
            pred_raw = clean_prediction(raw)
            pred = align_to_context(pred_raw, s["context"]) if use_span else pred_raw
            preds[j] = {
                "id": s.get("id", ""),
                "uit_id": s.get("uit_id", ""),
                "question": s.get("question", ""),
                "is_impossible": s.get("is_impossible"),
                "has_label": s.get("has_label", True),
                "ground_truth": s.get("answer"),
                "prediction_raw": pred_raw,
                "prediction": pred,
            }
        done += len(batch_idx)
        if start == 0 or done % max(args.log_every, bs) < bs or done == total:
            el = time.time() - t0
            rate = done / max(el, 1e-3)
            eta = (total - done) / max(rate, 1e-3)
            print(f"[Sub] {done}/{total} | {el/60:.1f}m | ETA {eta/60:.1f}m | {rate:.2f} sample/s", flush=True)

    out_path = Path(args.output)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False)
    print(f"[Sub] Saved {len(preds)} predictions → {out_path}", flush=True)


if __name__ == "__main__":
    main()
'''


def _ensure_eval_infer_script():
    """LUÔN đồng bộ eval_infer_subprocess.py từ bản nhúng trong notebook (single source of truth).

    Notebook là nguồn chuẩn duy nhất → tránh lỗi 'file cũ trên máy remote chưa cập nhật'.
    Chỉ ghi lại khi nội dung khác (đỡ IO thừa).
    """
    script = Path("eval_infer_subprocess.py")
    old = script.read_text(encoding="utf-8") if script.exists() else None
    if old != _EVAL_INFER_SUBPROCESS_SRC:
        script.write_text(_EVAL_INFER_SUBPROCESS_SRC, encoding="utf-8")
        print(f"[Infer] Đồng bộ {script.resolve()} từ bản nhúng notebook.", flush=True)
    return script


def _infer_predictions_subprocess(method_name, adapter_path, samples, max_seq_length, *, log_every=50, desc=None):
    """Inference bằng PROCESS riêng (transformers thuần, KHÔNG unsloth) → trả về preds list.

    Dùng chung cho metric eval lẫn submission export: trả về đúng cấu trúc như
    _infer_predictions in-kernel (list dict có id/prediction/...).
    """
    import shutil
    import subprocess
    import sys
    import tempfile

    script = _ensure_eval_infer_script()
    tmpdir = tempfile.mkdtemp(prefix="eval_infer_")
    samples_path = str(Path(tmpdir) / "samples.json")
    preds_path = str(Path(tmpdir) / "preds.json")
    with open(samples_path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False)

    cmd = [
        sys.executable, str(script),
        "--adapter-dir", str(adapter_path),
        "--samples-json", samples_path,
        "--output", preds_path,
        "--base-model", BASE_MODEL_NAME,
        "--max-seq-length", str(max_seq_length),
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--batch-size", str(INFER_BATCH_SIZE),
        "--log-every", str(log_every),
    ]
    if not USE_SPAN_POSTPROCESS:
        cmd.append("--no-span-postprocess")

    print(f"[Infer] {method_name}: tách process (transformers thuần, KHÔNG unsloth)", flush=True)
    print("  " + " ".join(cmd), flush=True)
    # Stream stdout process con vào cell (Jupyter không tự hiện fd của subprocess).
    proc = subprocess.Popen(
        cmd, env=dict(os.environ),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line.rstrip("\n"), flush=True)
    finally:
        proc.wait()

    if proc.returncode != 0:
        shutil.rmtree(tmpdir, ignore_errors=True)
        raise RuntimeError(f"{method_name}: subprocess inference lỗi (exit {proc.returncode}) — xem log phía trên.")
    if not Path(preds_path).exists():
        shutil.rmtree(tmpdir, ignore_errors=True)
        raise RuntimeError(f"{method_name}: subprocess xong nhưng không thấy {preds_path}")

    with open(preds_path, encoding="utf-8") as f:
        preds = json.load(f)
    shutil.rmtree(tmpdir, ignore_errors=True)
    print(f"[Infer] {method_name}: nhận {len(preds)} predictions từ subprocess.", flush=True)
    return preds


def export_submission_one_adapter(method_name, adapter_path, test_samples, max_seq_length):
    print(f"\n--- Submission export: {method_name} | {adapter_path} ---", flush=True)

    samples = test_samples
    if SUBMISSION_MAX_SAMPLES:
        samples = test_samples[:SUBMISSION_MAX_SAMPLES]
        print(f"[Submit] DEBUG mode: chỉ {len(samples)}/{len(test_samples)} câu", flush=True)
    else:
        print(f"[Submit] Full test split: {len(samples)} câu → results.json", flush=True)

    preds = _infer_predictions(
        method_name, adapter_path, samples, max_seq_length,
        log_every=SUBMISSION_LOG_EVERY, desc=f"Submit-{method_name}",
    )
    if preds is None:
        return None

    results = predictions_to_submission_dict(preds)
    if len(results) != len(samples):
        raise RuntimeError(f"{method_name}: results.json thiếu/trùng key ({len(results)} != {len(samples)})")
    if any(not isinstance(k, str) or not k for k in results):
        raise RuntimeError("results.json có uit_id rỗng hoặc không phải string")
    if any(not isinstance(v, str) for v in results.values()):
        raise RuntimeError("results.json phải có dạng {uit_id: answer_string}")
    if SUBMISSION_MAX_SAMPLES is None and len(results) != EXPECTED_TEST_SIZE:
        raise RuntimeError(f"Full submission phải có {EXPECTED_TEST_SIZE} keys, nhận {len(results)}")

    out_dir = Path(ADAPTER_SUBMISSION_DIRS.get(method_name, adapter_path))
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "results.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    # Read-back validation: bảo đảm file cuối đúng schema tham chiếu của adapter khác.
    saved_results = json.load(open(out_path, encoding="utf-8"))
    if saved_results != results:
        raise RuntimeError(f"Read-back validation thất bại: {out_path}")
    n_empty = sum(1 for v in results.values() if v == "")
    print(f"[Submit] Saved {out_path} | total={len(results)} | noAns={n_empty} | hasAns={len(results)-n_empty}")
    print("[Submit] Schema OK: {\"uit_id\": \"answer_of_model\"}")
    return out_path

In [ ]:
variant_map = {v["name"]: v for v in ADAPTER_VARIANTS}

# ===== 1) Metric eval (validation, có GT) =====
if RUN_METRIC_EVAL:
    if "eval_samples" not in globals():
        raise NameError("Chạy cell tải dataset trước (cần eval_samples).")
    all_results = {}

    for method in EVAL_METHODS:
        path = variant_map[method]["save_path"]
        result = eval_one_adapter(method, path, eval_samples, max_seq_length)
        if result is not None:
            all_results[method] = result

    line = "=" * 72
    print("\n" + line)
    print(f"  SO SÁNH ADAPTERS — UIT-ViQuAD2.0 ({EVAL_SPLIT}) | Qwen2.5-1.5B-Instruct")
    print(line)
    print(f"{'Method':<12} {'HasAns EM':>12} {'HasAns F1':>12} {'NoAns Acc':>12} {'n_has/n_no':>14}")
    print("-" * 72)
    for method, result in all_results.items():
        m = result["metrics"]
        noans = f"{m['noans_accuracy']:.2f}%" if m["noans_accuracy"] is not None else "N/A"
        print(f"{method:<12} {m['hasans_em']:>11.2f}% {m['hasans_f1']:>11.2f}% {noans:>12} "
              f"{m['n_hasans']}/{m['n_noans']:>6}")
    print(line)

    for method, result in all_results.items():
        m = result["metrics"]
        has_f1 = [p["f1"] for p in result["predictions"] if p.get("f1") is not None and not p.get("is_impossible")]
        buckets = {"0-0.2": 0, "0.2-0.5": 0, "0.5-0.8": 0, "0.8-1.0": 0, "1.0": 0}
        for f1 in has_f1:
            if f1 == 1.0: buckets["1.0"] += 1
            elif f1 >= 0.8: buckets["0.8-1.0"] += 1
            elif f1 >= 0.5: buckets["0.5-0.8"] += 1
            elif f1 >= 0.2: buckets["0.2-0.5"] += 1
            else: buckets["0-0.2"] += 1

        print(f"\n--- {method.upper()} detail ---")
        print(f"Exact Match (HasAns): {m['hasans_em']:.2f}%")
        print(f"F1-score (HasAns)   : {m['hasans_f1']:.2f}%")
        if m["n_hasans"]:
            print("Phân phối F1-score (HasAns):")
            for b, c in buckets.items():
                print(f"  F1 = {b}: {c} câu ({100*c/m['n_hasans']:.1f}%)")

    save_payload = {
        "dataset": DATASET_NAME,
        "eval_split": EVAL_SPLIT,
        "base_model": BASE_MODEL_NAME,
        "max_seq_length": max_seq_length,
        "train_common": TRAIN_COMMON,
        "summary": {k: v["metrics"] for k, v in all_results.items()},
        "predictions": {k: v["predictions"] for k, v in all_results.items()},
    }
    with open(COMPARE_EVAL_PATH, "w", encoding="utf-8") as f:
        json.dump(save_payload, f, ensure_ascii=False, indent=2)
    print(f"\nSaved comparison → {COMPARE_EVAL_PATH}")
else:
    print("RUN_METRIC_EVAL=False — bỏ qua metric eval.")

# ===== 2) Submission export (FULL test ~7301 → results.json) =====
if RUN_SUBMISSION_EXPORT:
    if "test_samples" not in globals():
        raise NameError("Chạy cell tải dataset trước (cần test_samples).")
    expected = len(test_samples) if not SUBMISSION_MAX_SAMPLES else min(SUBMISSION_MAX_SAMPLES, len(test_samples))
    print(f"\n{'='*72}\n  EXPORT results.json — test split ({expected} câu / {len(test_samples)} total)\n{'='*72}")

    for method in EVAL_METHODS:
        path = variant_map[method]["save_path"]
        export_submission_one_adapter(method, path, test_samples, max_seq_length)
else:
    print("RUN_SUBMISSION_EXPORT=False — bỏ qua results.json.")